
# Paper 4 — Notebook 07: final validation and corrected analyses

This notebook is designed to be run **after** `Paper 4_clean_and_split.ipynb` in the **same Colab runtime**, so that:

- `/content/artifacts/reg_clean.parquet`
- `/content/artifacts/cls_clean.parquet`
- `/content/artifacts/feature_sets.json`
- `/content/artifacts/meta.json`

already exist.

It performs three submission-critical analyses:

1. **Corrected process maps** — recomputes both linear and volumetric energy-density features at every power–velocity grid point.
2. **Classification robustness** — source-by-class support table plus leave-one-source-out (LOSO) nested evaluation.
3. **Corrected Eagar–Tsai comparison** — material-specific melting-temperature audit, separate raw/calibrated E-T, both RF and XGBoost, and source-level bootstrap confidence intervals.

Outputs are written to `/content/results_final/` so the older result files are not overwritten.

**Important:** the numbers produced here should become the authoritative numbers for the next manuscript revision. Do not force the code to reproduce an older table if the corrected analysis changes the values.


In [ ]:

# ============================================================
# CELL 1 — SELF-CONTAINED SETUP
# ============================================================
# If /content/artifacts does not exist, this cell recreates the SAME frozen
# cleaning/split artifacts used by the earlier Paper 4 notebooks by cloning
# the official MeltpoolNet repository.

import os, re, json, time, warnings, itertools, math, subprocess, sys, importlib.util
import numpy as np
import pandas as pd
warnings.filterwarnings("ignore")

# Install only missing packages.
PKGS = {
    "pandas":"pandas", "numpy":"numpy", "scikit-learn":"sklearn",
    "pyarrow":"pyarrow", "xgboost":"xgboost", "matplotlib":"matplotlib"
}
missing_pkgs = [p for p, imp in PKGS.items() if importlib.util.find_spec(imp) is None]
if missing_pkgs:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing_pkgs])

from sklearn.model_selection import (
    KFold, GroupKFold, StratifiedKFold, StratifiedGroupKFold
)
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import NearestNeighbors
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import (
    r2_score, mean_absolute_error, f1_score, balanced_accuracy_score,
    matthews_corrcoef, accuracy_score
)
from xgboost import XGBRegressor, XGBClassifier

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

SEED = 42
np.random.seed(SEED)

ART = "/content/artifacts"
RES = "/content/results_final"
os.makedirs(ART, exist_ok=True)
os.makedirs(RES, exist_ok=True)

required = [
    f"{ART}/reg_clean.parquet",
    f"{ART}/cls_clean.parquet",
    f"{ART}/feature_sets.json",
    f"{ART}/meta.json",
]

# ------------------------------------------------------------------
# Recreate the exact frozen artifacts if this is a fresh Colab runtime.
# ------------------------------------------------------------------
if not all(os.path.exists(p) for p in required):
    print("Frozen artifacts not found. Recreating them from MeltpoolNet...")

    repo = "/content/MeltpoolNet"
    reg_path = f"{repo}/Data/meltpoolnet_regression.csv"
    cls_path = f"{repo}/Data/meltpoolnet_classification.csv"

    if not os.path.exists(reg_path):
        subprocess.run(
            ["git", "clone", "--depth", "1",
             "https://github.com/BaratiLab/MeltpoolNet.git", repo],
            check=True
        )

    reg0 = pd.read_csv(reg_path)
    cls0 = pd.read_csv(cls_path)

    def drop_junk(df):
        junk = [c for c in df.columns if c.startswith("Unnamed:") or str(c).strip() == ""]
        if "comment" in df.columns:
            junk.append("comment")
        return df.drop(columns=junk)

    reg0 = drop_junk(reg0)
    cls0 = drop_junk(cls0)

    CLASSES0 = ["desirable", "keyhole", "LOF", "balling"]
    CLS_ID0 = {c:i for i,c in enumerate(CLASSES0)}

    cls0 = cls0[cls0["meltpool shape"].isin(CLASSES0)].copy()
    cls0["y_class"] = cls0["meltpool shape"].map(CLS_ID0).astype(int)

    def comp_cols(df):
        return [c for c in df.columns if re.search(r"\(wt\.?%\)", c)]

    reg_F1 = ["Power","Velocity","powder flowrate","layer thickness","beam D","Hatch spacing"]
    reg_F2 = reg_F1 + ["density","Cp","k","melting T","absorption coefficient","minimum absorptivity"]
    reg_F3 = reg_F2 + ["E (J/mm)","E (J/mm3)"]
    reg_F4 = reg_F3 + comp_cols(reg0)
    reg_F5 = reg_F3 + ["Material"]

    cls_F1 = ["Power","Velocity","Hatch spacing","layer thickness","beam D"]
    cls_F2 = cls_F1 + ["density","Cp","k","melting T","absorption coefficient","minimal absorptivity"]
    cls_F3 = cls_F2 + ["p/lb","p/l","p/b2","p/b","vb","vl"]
    cls_F4 = cls_F3 + comp_cols(cls0)
    cls_F5 = cls_F3 + ["Material"]

    REG_FEATURES0 = {"F1":reg_F1,"F2":reg_F2,"F3":reg_F3,"F4":reg_F4,"F5":reg_F5}
    CLS_FEATURES0 = {"F1":cls_F1,"F2":cls_F2,"F3":cls_F3,"F4":cls_F4,"F5":cls_F5}

    def coerce_numeric(df, cols):
        for c in cols:
            if c != "Material":
                df[c] = pd.to_numeric(df[c], errors="coerce")
        return df

    reg0 = coerce_numeric(reg0, sorted(set(sum(REG_FEATURES0.values(), []))))
    cls0 = coerce_numeric(cls0, sorted(set(sum(CLS_FEATURES0.values(), []))))

    reg0 = reg0.reset_index(drop=True)
    cls0 = cls0.reset_index(drop=True)

    reg0["has_depth"] = reg0["depth of meltpool"].notna()
    reg0["has_width"] = reg0["width of melt pool"].notna()
    reg0["has_class"] = reg0["meltpool shape"].isin(CLASSES0)
    reg0["y_class"] = reg0["meltpool shape"].map(CLS_ID0)

    K_REG, K_CLS = 10, 5

    def reg_splits(mask_col, k=K_REG):
        mask = reg0[mask_col].values
        idx = np.where(mask)[0]
        groups0 = reg0.loc[idx, "paper ID"].values
        v0 = np.full(len(reg0), -1, dtype=int)
        v1 = np.full(len(reg0), -1, dtype=int)

        for f, (_, te) in enumerate(
            KFold(k, shuffle=True, random_state=SEED).split(idx)
        ):
            v0[idx[te]] = f

        for f, (_, te) in enumerate(
            GroupKFold(n_splits=k).split(idx, groups=groups0)
        ):
            v1[idx[te]] = f
        return v0, v1

    reg0["v0_depth"], reg0["v1_depth"] = reg_splits("has_depth")
    reg0["v0_width"], reg0["v1_width"] = reg_splits("has_width")

    def lomo_materials(mask_col, min_rows=40):
        vc = reg0.loc[reg0[mask_col], "Material"].value_counts()
        return vc[vc >= min_rows].index.tolist()

    LOMO_DEPTH = lomo_materials("has_depth")
    LOMO_WIDTH = lomo_materials("has_width")

    cy = cls0["y_class"].values
    cg = cls0["paper ID"].values
    cls0["v0"] = -1
    cls0["v1"] = -1

    for f, (_, te) in enumerate(
        StratifiedKFold(K_CLS, shuffle=True, random_state=SEED).split(cls0, cy)
    ):
        cls0.loc[te, "v0"] = f

    for f, (_, te) in enumerate(
        StratifiedGroupKFold(
            K_CLS, shuffle=True, random_state=SEED
        ).split(cls0, cy, groups=cg)
    ):
        cls0.loc[te, "v1"] = f

    LOMO_CLS = cls0["Material"].value_counts()
    LOMO_CLS = LOMO_CLS[LOMO_CLS >= 40].index.tolist()

    reg0.to_parquet(f"{ART}/reg_clean.parquet", index=False)
    cls0.to_parquet(f"{ART}/cls_clean.parquet", index=False)

    feature_sets0 = {
        "regression": REG_FEATURES0,
        "classification": CLS_FEATURES0
    }
    with open(f"{ART}/feature_sets.json", "w") as f:
        json.dump(feature_sets0, f, indent=2)

    commit = subprocess.run(
        ["git", "-C", repo, "rev-parse", "HEAD"],
        capture_output=True, text=True
    ).stdout.strip()

    meta0 = {
        "seed": SEED,
        "source_key": "paper ID",
        "classes": CLASSES0,
        "regression": {
            "primary_target": "depth of meltpool",
            "secondary_target": "width of melt pool",
            "appendix_target": "length of melt pool",
            "k_folds": K_REG,
            "n_depth": int(reg0["has_depth"].sum()),
            "n_width": int(reg0["has_width"].sum()),
            "n_class_labels": int(reg0["has_class"].sum()),
            "n_depth_and_width": int((reg0["has_depth"] & reg0["has_width"]).sum()),
            "lomo_depth_materials": LOMO_DEPTH,
            "lomo_width_materials": LOMO_WIDTH,
        },
        "classification": {
            "k_folds": K_CLS,
            "class_counts": {
                c:int((cls0["y_class"] == i).sum())
                for c,i in CLS_ID0.items()
            },
            "lomo_materials": LOMO_CLS,
            "note": (
                "grouped folds strand balling in some folds; "
                "use pooled out-of-fold predictions for macro-F1."
            )
        },
        "data_commit": commit,
    }
    with open(f"{ART}/meta.json", "w") as f:
        json.dump(meta0, f, indent=2)

    print("Frozen artifacts recreated.")

# ------------------------------------------------------------------
# Load frozen artifacts
# ------------------------------------------------------------------
reg = pd.read_parquet(f"{ART}/reg_clean.parquet")
cls = pd.read_parquet(f"{ART}/cls_clean.parquet")
with open(f"{ART}/feature_sets.json") as f:
    FS = json.load(f)
with open(f"{ART}/meta.json") as f:
    META = json.load(f)

F3_REG = FS["regression"]["F3"]
F3_CLS = FS["classification"]["F3"]
CLASSES = META["classes"]

print("\nFrozen dataset ready")
print("  regression rows       :", len(reg))
print("  depth-labelled rows   :", int(reg["has_depth"].sum()))
print("  classification rows   :", len(cls))
print("  classification studies:", cls["paper ID"].nunique())
print("  data commit           :", META.get("data_commit"))
print("  results directory     :", RES)


In [ ]:

# ============================================================
# CELL 2 — CORRECTED PROCESS MAPS
# ============================================================
# Corrections relative to Notebook 05:
#   1) E (J/mm) is recomputed at every grid point.
#   2) E (J/mm3) is ALSO recomputed at every grid point:
#      P / [v * hatch(mm) * layer(mm)]
#      because hatch spacing and layer thickness are stored in micrometres.
#   3) The plotted grid uses the 5th–95th percentile power/velocity envelope.
#      This avoids allowing a handful of extreme/unit-suspect tail values to define
#      the visualization while remaining strictly inside the observed range.
#
# The TRAINING feature table remains the frozen F3 benchmark so this analysis
# remains comparable with the manuscript's existing model definition.

sub = reg[reg["has_depth"]].reset_index(drop=True).copy()

# Same absorptivity sanitization used in the physical-audit sensitivity analysis.
a = pd.to_numeric(sub["absorption coefficient"], errors="coerce")
sub["absorption coefficient"] = np.where((a > 0) & (a <= 1), a, np.nan)

# Process maps are restricted to powder-bed-fusion records.
sub_map = sub[sub["Process"].eq("PBF")].copy()

# Diagnostic: show very low PBF velocities because the literature-aggregated
# benchmark contains some source-unit heterogeneity.
low_v = sub_map[pd.to_numeric(sub_map["Velocity"], errors="coerce") < 10]
if len(low_v):
    print("NOTE: PBF rows with Velocity < 10 in the frozen benchmark:", len(low_v))
    print(low_v[["Material","Power","Velocity","paper ID"]].drop_duplicates().to_string(index=False))
    print("The map grid therefore uses a 5th–95th percentile envelope rather than the extreme tails.\n")

def recompute_grid_energy_features(G):
    """Recompute every power/velocity-derived F3 feature used by the regression model."""
    G = G.copy()
    P = pd.to_numeric(G["Power"], errors="coerce")
    V = pd.to_numeric(G["Velocity"], errors="coerce")
    H_um = pd.to_numeric(G["Hatch spacing"], errors="coerce")
    T_um = pd.to_numeric(G["layer thickness"], errors="coerce")

    if (V <= 0).any():
        raise ValueError("Non-positive velocity encountered in process-map grid.")
    if H_um.isna().any() or T_um.isna().any():
        raise ValueError("Hatch spacing/layer thickness median is missing for this alloy.")
    if (H_um <= 0).any() or (T_um <= 0).any():
        raise ValueError("Non-positive hatch spacing/layer thickness encountered.")

    # Power [W] / velocity [mm/s] = J/mm
    G["E (J/mm)"] = P / V

    # Hatch and layer thickness are stored in µm -> convert to mm.
    H_mm = H_um / 1000.0
    T_mm = T_um / 1000.0
    G["E (J/mm3)"] = P / (V * H_mm * T_mm)
    return G

def build_corrected_map(alloy, n_grid=60, n_ens=5, q_lo=0.05, q_hi=0.95):
    d = sub_map[sub_map["Material"].eq(alloy)].copy()
    if len(d) < 40:
        return None

    P = pd.to_numeric(d["Power"], errors="coerce")
    V = pd.to_numeric(d["Velocity"], errors="coerce")
    ok = P.notna() & V.notna() & (P > 0) & (V > 0)
    d = d.loc[ok].reset_index(drop=True)
    P = P.loc[ok].reset_index(drop=True)
    V = V.loc[ok].reset_index(drop=True)

    if len(d) < 40:
        return None

    y = np.log10(d["depth of meltpool"].astype(float).values)
    X = d[F3_REG].copy()

    # Full-data models used only to draw the map.
    models = [
        XGBRegressor(
            random_state=s, n_estimators=300, max_depth=5, learning_rate=0.08,
            subsample=0.8, colsample_bytree=0.8, verbosity=0, n_jobs=-1
        ).fit(X, y)
        for s in range(n_ens)
    ]

    # Robust interior of the observed power–velocity envelope.
    pg = np.linspace(P.quantile(q_lo), P.quantile(q_hi), n_grid)
    vg = np.linspace(V.quantile(q_lo), V.quantile(q_hi), n_grid)
    PP, VV = np.meshgrid(pg, vg)

    # Hold independent nonswept features at alloy-wise medians.
    base = d[F3_REG].median(numeric_only=True).reindex(F3_REG)
    for needed in ["Hatch spacing", "layer thickness"]:
        if not np.isfinite(base[needed]) or base[needed] <= 0:
            raise ValueError(f"{alloy}: no valid positive median for {needed}")

    G = pd.DataFrame(np.tile(base.values, (PP.size, 1)), columns=F3_REG)
    G["Power"] = PP.ravel()
    G["Velocity"] = VV.ravel()
    G = recompute_grid_energy_features(G)

    # Hard consistency checks: these are the two issues the prior map failed.
    chk_lin = np.allclose(
        G["E (J/mm)"].values,
        G["Power"].values / G["Velocity"].values,
        rtol=1e-10, atol=1e-12
    )
    chk_vol = np.allclose(
        G["E (J/mm3)"].values,
        G["Power"].values / (
            G["Velocity"].values *
            (G["Hatch spacing"].values / 1000.0) *
            (G["layer thickness"].values / 1000.0)
        ),
        rtol=1e-10, atol=1e-12
    )
    assert chk_lin and chk_vol, "Derived grid-energy consistency check failed."

    Pm = np.vstack([m.predict(G) for m in models])
    mu = Pm.mean(axis=0).reshape(PP.shape)
    sd = Pm.std(axis=0).reshape(PP.shape)

    # -------- out-of-fold disagreement threshold --------
    groups = d["paper ID"].values
    n_splits = min(5, max(2, len(np.unique(groups))))
    oof_sd = np.full(len(d), np.nan)

    for tr_i, te_i in GroupKFold(n_splits=n_splits).split(
        np.arange(len(d)), groups=groups
    ):
        fold_preds = np.vstack([
            XGBRegressor(
                random_state=s, n_estimators=300, max_depth=5, learning_rate=0.08,
                subsample=0.8, colsample_bytree=0.8, verbosity=0, n_jobs=-1
            ).fit(X.iloc[tr_i], y[tr_i]).predict(X.iloc[te_i])
            for s in range(n_ens)
        ])
        oof_sd[te_i] = fold_preds.std(axis=0)

    sd_thr = float(np.nanquantile(oof_sd, 0.75))

    # -------- input-space support threshold --------
    imp = SimpleImputer(strategy="median").fit(X)
    X_imp = imp.transform(X)
    sc = StandardScaler().fit(X_imp)
    X_std = sc.transform(X_imp)

    # Six neighbours here because the first neighbour of a training row is itself.
    nn_train = NearestNeighbors(n_neighbors=6).fit(X_std)
    dtr, _ = nn_train.kneighbors(X_std, n_neighbors=6)
    dist_thr = float(np.quantile(dtr[:, 1:].mean(axis=1), 0.90))

    G_std = sc.transform(imp.transform(G))
    dg, _ = nn_train.kneighbors(G_std, n_neighbors=5)
    dist = dg.mean(axis=1).reshape(PP.shape)

    mask = (sd > sd_thr) | (dist > dist_thr)

    return {
        "alloy": alloy,
        "PP": PP, "VV": VV, "mu": mu, "sd": sd, "mask": mask,
        "P": P.values, "V": V.values,
        "n": len(d),
        "sd_thr": sd_thr,
        "dist_thr": dist_thr,
        "pct_masked": float(mask.mean() * 100),
        "grid_P_min": float(pg.min()), "grid_P_max": float(pg.max()),
        "grid_V_min": float(vg.min()), "grid_V_max": float(vg.max()),
        "median_hatch_um": float(base["Hatch spacing"]),
        "median_layer_um": float(base["layer thickness"]),
        "grid_EDvol_min": float(G["E (J/mm3)"].min()),
        "grid_EDvol_max": float(G["E (J/mm3)"].max()),
    }

map_alloys = ["Ti-6Al-4V", "SS316L", "IN718"]
maps = [build_corrected_map(a) for a in map_alloys]
maps = [m for m in maps if m is not None]

print("\nCORRECTED MAP SUMMARY")
for M in maps:
    print(
        f"{M['alloy']:12s} n={M['n']:4d} | outside-domain={M['pct_masked']:5.1f}% "
        f"| P={M['grid_P_min']:.1f}–{M['grid_P_max']:.1f} W "
        f"| V={M['grid_V_min']:.1f}–{M['grid_V_max']:.1f} mm/s "
        f"| E_vol={M['grid_EDvol_min']:.2f}–{M['grid_EDvol_max']:.2f} J/mm³"
    )

coverage = pd.DataFrame([{
    "alloy": M["alloy"],
    "n_training_rows": M["n"],
    "pct_grid_outside_domain": round(M["pct_masked"], 1),
    "grid_power_min_W": M["grid_P_min"],
    "grid_power_max_W": M["grid_P_max"],
    "grid_velocity_min_mm_s": M["grid_V_min"],
    "grid_velocity_max_mm_s": M["grid_V_max"],
    "median_hatch_um": M["median_hatch_um"],
    "median_layer_um": M["median_layer_um"],
    "grid_E_J_mm3_min": M["grid_EDvol_min"],
    "grid_E_J_mm3_max": M["grid_EDvol_max"],
} for M in maps])
coverage.to_csv(f"{RES}/process_map_coverage_corrected.csv", index=False)

# Plot and export both vector PDF and high-resolution PNG.
fig, axes = plt.subplots(1, len(maps), figsize=(5.3 * len(maps), 4.5))
if len(maps) == 1:
    axes = [axes]

for ax, M in zip(axes, maps):
    im = ax.pcolormesh(M["PP"], M["VV"], M["mu"], shading="auto", cmap="viridis")
    ax.contourf(
        M["PP"], M["VV"], M["mask"].astype(float),
        levels=[0.5, 1.5], colors="none", hatches=["xxx"]
    )
    ax.contour(
        M["PP"], M["VV"], M["mask"].astype(float),
        levels=[0.5], colors="white", linewidths=1.0
    )
    ax.scatter(
        M["P"], M["V"], s=7, c="white", edgecolors="black",
        linewidths=0.3, alpha=0.7, label="training data"
    )
    ax.set_xlim(M["PP"].min(), M["PP"].max())
    ax.set_ylim(M["VV"].min(), M["VV"].max())
    ax.set_xlabel("Power (W)")
    ax.set_ylabel("Velocity (mm/s)")
    ax.set_title(
        f"{M['alloy']} (n={M['n']})\n"
        f"hatched = outside domain ({M['pct_masked']:.0f}%)",
        fontsize=10
    )
    ax.legend(fontsize=7, loc="upper left")
    plt.colorbar(im, ax=ax, label="predicted log₁₀(depth µm)")

plt.tight_layout()
plt.savefig(f"{RES}/fig_process_maps_corrected.pdf", bbox_inches="tight")
plt.savefig(f"{RES}/fig_process_maps_corrected.png", dpi=600, bbox_inches="tight")
plt.show()

print("\nSaved:")
print(f"  {RES}/process_map_coverage_corrected.csv")
print(f"  {RES}/fig_process_maps_corrected.pdf")
print(f"  {RES}/fig_process_maps_corrected.png")


In [ ]:

# ============================================================
# CELL 3 — CLASSIFICATION SOURCE SUPPORT + LOSO ROBUSTNESS
# ============================================================
# Goal:
#   A) quantify which classes are represented by each of the 16 source studies;
#   B) replace the fragile 5-fold sensitivity check with leave-one-source-out (LOSO);
#   C) keep hyperparameter selection nested inside the 15 training sources.
#
# We use XGBoost here because it is the best V1 classifier in the existing baseline table.
# This is a robustness/sensitivity analysis, not a replacement for the four-model baseline table.

X = cls[F3_CLS].copy()
y = cls["y_class"].astype(int).values
groups = cls["paper ID"].astype(str).values
class_ids = np.arange(len(CLASSES))

# ---------- source × class support table ----------
support = pd.crosstab(cls["paper ID"], cls["meltpool shape"])
support = support.reindex(columns=CLASSES, fill_value=0)
support["Total"] = support.sum(axis=1)
support["fraction_of_dataset"] = support["Total"] / len(cls)
support = support.sort_values("Total", ascending=False)

support.to_csv(f"{RES}/classification_source_by_class.csv")

print("SOURCE × CLASS SUPPORT")
print(support.to_string())
print("\nStudies containing each class:")
for c in CLASSES:
    print(f"  {c:10s}: {(support[c] > 0).sum()} / {len(support)} studies")

# Check that every LOSO training set still contains all four classes.
for src in np.unique(groups):
    train_classes = np.unique(y[groups != src])
    if set(train_classes) != set(class_ids):
        raise RuntimeError(
            f"Holding out source {src} removes a class from training. "
            f"Training classes = {train_classes}"
        )
print("\nEvery LOSO training set retains all four classes.")

# ---------- nested XGBoost tuning ----------
PARAM_GRID = [
    {"max_depth": md, "learning_rate": lr, "subsample": 0.8}
    for md in [3, 6]
    for lr in [0.05, 0.10]
]

def make_xgb_classifier(params, seed=SEED):
    return XGBClassifier(
        random_state=seed,
        n_estimators=400,
        max_depth=params["max_depth"],
        learning_rate=params["learning_rate"],
        subsample=params["subsample"],
        objective="multi:softprob",
        num_class=len(CLASSES),
        eval_metric="mlogloss",
        verbosity=0,
        n_jobs=-1,
    )

def tune_xgb_grouped(Xtr, ytr, gtr):
    """Inner 3-fold stratified-group tuning on the OUTER training sources only."""
    inner = StratifiedGroupKFold(
        n_splits=3, shuffle=True, random_state=SEED
    )
    best_score = -np.inf
    best_params = None

    for params in PARAM_GRID:
        inner_oof = np.full(len(ytr), -1, dtype=int)

        for itr, ite in inner.split(Xtr, ytr, groups=gtr):
            # Guard against an impossible class-missing inner train split.
            if len(np.unique(ytr[itr])) < len(CLASSES):
                inner_oof[ite] = -1
                continue

            mdl = make_xgb_classifier(params)
            mdl.fit(Xtr.iloc[itr], ytr[itr])
            inner_oof[ite] = mdl.predict(Xtr.iloc[ite]).astype(int)

        ok = inner_oof >= 0
        if ok.sum() == 0:
            continue

        score = f1_score(
            ytr[ok], inner_oof[ok],
            labels=class_ids, average="macro", zero_division=0
        )
        if score > best_score:
            best_score = score
            best_params = params.copy()

    if best_params is None:
        raise RuntimeError("No valid inner grouped tuning configuration.")
    return best_params, float(best_score)

loso_pred = np.full(len(cls), -1, dtype=int)
tuning_rows = []

sources = np.unique(groups)
t0 = time.time()

for i, src in enumerate(sources, 1):
    te = groups == src
    tr = ~te

    Xtr = X.loc[tr].reset_index(drop=True)
    ytr = y[tr]
    gtr = groups[tr]

    best_params, inner_f1 = tune_xgb_grouped(Xtr, ytr, gtr)
    mdl = make_xgb_classifier(best_params)
    mdl.fit(Xtr, ytr)

    pred = mdl.predict(X.loc[te]).astype(int)
    loso_pred[te] = pred

    tuning_rows.append({
        "held_out_source": src,
        "n_test": int(te.sum()),
        "inner_macroF1": inner_f1,
        **best_params,
    })
    print(
        f"[{i:02d}/{len(sources)}] source={src:>8s} n={te.sum():4d} "
        f"inner-F1={inner_f1:.3f} params={best_params}"
    )

assert (loso_pred >= 0).all(), "Some LOSO rows were not predicted."

print(f"\nLOSO completed in {(time.time()-t0)/60:.1f} min")

# ---------- pooled OOF metrics ----------
pooled = {
    "macroF1": f1_score(
        y, loso_pred, labels=class_ids, average="macro", zero_division=0
    ),
    "balanced_accuracy": balanced_accuracy_score(y, loso_pred),
    "MCC": matthews_corrcoef(y, loso_pred),
    "accuracy": accuracy_score(y, loso_pred),
}
print("\nPOOLED LOSO METRICS")
for k, v in pooled.items():
    print(f"  {k:18s}: {v:.3f}")

# ---------- per-source diagnostics ----------
per_source = []
for src in sources:
    idx = groups == src
    present = np.unique(y[idx])

    per_source.append({
        "paper_ID": src,
        "n": int(idx.sum()),
        "n_true_classes": int(len(present)),
        # All-four metric is intentionally strict.
        "macroF1_all4": float(f1_score(
            y[idx], loso_pred[idx],
            labels=class_ids, average="macro", zero_division=0
        )),
        # Present-class metric is easier to interpret for sources lacking some classes.
        "macroF1_present": float(f1_score(
            y[idx], loso_pred[idx],
            labels=present, average="macro", zero_division=0
        )),
        "accuracy": float(accuracy_score(y[idx], loso_pred[idx])),
    })

per_source_df = pd.DataFrame(per_source).sort_values("n", ascending=False)
per_source_df.to_csv(f"{RES}/classification_loso_by_source.csv", index=False)

# ---------- source-level bootstrap confidence intervals ----------
def bootstrap_source_metric(y_true, y_pred, src, metric_fn, B=2000, seed=SEED):
    rng = np.random.default_rng(seed)
    unique_src = np.unique(src)
    vals = []

    src_to_idx = {s: np.where(src == s)[0] for s in unique_src}

    for _ in range(B):
        sampled = rng.choice(unique_src, size=len(unique_src), replace=True)
        idx = np.concatenate([src_to_idx[s] for s in sampled])

        try:
            val = metric_fn(y_true[idx], y_pred[idx])
            if np.isfinite(val):
                vals.append(float(val))
        except Exception:
            pass

    return (
        float(np.percentile(vals, 2.5)),
        float(np.percentile(vals, 97.5)),
        len(vals)
    )

metric_fns = {
    "macroF1": lambda a,b: f1_score(
        a,b,labels=class_ids,average="macro",zero_division=0
    ),
    "balanced_accuracy": balanced_accuracy_score,
    "MCC": matthews_corrcoef,
    "accuracy": accuracy_score,
}

summary_rows = []
for name, fn in metric_fns.items():
    lo, hi, nb = bootstrap_source_metric(y, loso_pred, groups, fn)
    summary_rows.append({
        "metric": name,
        "estimate": pooled[name],
        "CI95_lo": lo,
        "CI95_hi": hi,
        "bootstrap_replicates_used": nb,
    })

summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(f"{RES}/classification_loso_summary.csv", index=False)

# Row-level predictions for complete reproducibility.
pred_df = cls[["paper ID","Material","meltpool shape","y_class"]].copy()
pred_df["pred_class_id"] = loso_pred
pred_df["pred_class"] = [CLASSES[i] for i in loso_pred]
pred_df.to_csv(f"{RES}/classification_loso_predictions.csv", index=False)

pd.DataFrame(tuning_rows).to_csv(
    f"{RES}/classification_loso_tuning.csv", index=False
)

print("\nLOSO SUMMARY WITH SOURCE-LEVEL 95% BOOTSTRAP CIs")
print(summary_df.round(3).to_string(index=False))

# Supplementary diagnostic plot.
plot_df = per_source_df.sort_values("macroF1_present")
fig, ax = plt.subplots(figsize=(8, 5))
ax.barh(plot_df["paper_ID"].astype(str), plot_df["macroF1_present"])
ax.set_xlabel("LOSO macro-F1 over classes present in held-out source")
ax.set_ylabel("Held-out source study")
ax.set_xlim(0, 1)
ax.grid(axis="x", alpha=0.3)
plt.tight_layout()
plt.savefig(f"{RES}/fig_classification_loso.pdf", bbox_inches="tight")
plt.savefig(f"{RES}/fig_classification_loso.png", dpi=600, bbox_inches="tight")
plt.show()

print("\nSaved classification robustness outputs to:", RES)


In [ ]:

# ============================================================
# CELL 4 — CORRECTED EAGAR–TSAI + RF/XGB + SOURCE BOOTSTRAP
# ============================================================
# This cell deliberately does NOT use the obsolete global Tm > 900 K rule.
#
# The melting-temperature flags below reproduce the material-specific audit
# already documented in the manuscript draft / Supplement plan for this frozen
# MeltpoolNet commit:
#   - every exact 273 K entry is a placeholder-like value;
#   - AlSi10Mg 1142 K, HCP Cu 1631 K, Invar36 2000 K, MS1- 2848 K
#     are the audited material-inconsistent values.
#
# If Supplement S4 is later changed after checking primary material sources,
# update this frozen audit map and rerun this notebook.

# ---------- frozen material-specific audit decisions ----------
def melting_temperature_bad_mask(df):
    T = pd.to_numeric(df["melting T"], errors="coerce")
    M = df["Material"].astype(str)

    bad = np.isclose(T, 273.0, atol=1e-8, equal_nan=False)
    bad |= M.eq("AlSi10Mg") & np.isclose(T, 1142.0, atol=1e-8)
    bad |= M.eq("HCP Cu")   & np.isclose(T, 1631.0, atol=1e-8)
    bad |= M.eq("Invar36")  & np.isclose(T, 2000.0, atol=1e-8)
    bad |= M.eq("MS1-")     & np.isclose(T, 2848.0, atol=1e-8)
    return pd.Series(bad, index=df.index)

s = reg[reg["has_depth"]].reset_index(drop=True).copy()
mt_bad = melting_temperature_bad_mask(s)

audit_rows = (
    s.loc[mt_bad, ["Material","melting T"]]
     .value_counts()
     .reset_index(name="n_flagged")
     .sort_values(["Material","melting T"])
)
audit_rows.to_csv(f"{RES}/melting_temperature_audit_flags.csv", index=False)

print("Melting-temperature audit")
print("  flagged rows:", int(mt_bad.sum()))
print(audit_rows.to_string(index=False))
if int(mt_bad.sum()) != 98:
    print(
        "\nWARNING: the frozen dataset no longer reproduces the manuscript's 98-row audit. "
        "Do not submit until Supplement S4 and the code are reconciled."
    )

# Absorptivity treatment:
#   ML feature: values outside (0,1] -> NaN, as in the physical-audit model.
#   E-T heat input: use measured valid absorptivity; otherwise use a nominal default.
a = pd.to_numeric(s["absorption coefficient"], errors="coerce")
s["absorption coefficient"] = np.where((a > 0) & (a <= 1), a, np.nan)

ETA_DEFAULT = 0.35
eta_used = np.where((a > 0) & (a <= 1), a, ETA_DEFAULT)

# E-T also requires a physically interpretable beam diameter.
beam = pd.to_numeric(s["beam D"], errors="coerce")
beam_bad = beam.notna() & (beam < 1)
print("\nBeam-D rows < 1 µm excluded from the analytical comparison:", int(beam_bad.sum()))

# Diagnostic only: do not silently modify velocity here.
pbf_low_v = s[
    s["Process"].eq("PBF") &
    (pd.to_numeric(s["Velocity"], errors="coerce") < 10)
]
if len(pbf_low_v):
    print(
        "\nNOTE: the frozen benchmark contains PBF rows with Velocity < 10 in the stated mm/s column."
    )
    print(
        pbf_low_v[["Material","Power","Velocity","paper ID"]]
        .drop_duplicates().to_string(index=False)
    )
    print(
        "These rows are retained to preserve the frozen benchmark. "
        "They should be source-checked before making a strong absolute-physics claim."
    )

# ---------- Eagar–Tsai numerical solution ----------
NQ = 400
trap = np.trapezoid if hasattr(np, "trapezoid") else np.trapz

def et_dT_grid(w, z, P, v, sigma, rho, cp, k, eta):
    alpha = k / (rho * cp)
    q = eta * P
    tmax = max(
        50 * sigma**2 / alpha,
        50 * sigma / max(v, 1e-9),
        1e-4
    )
    tau = np.logspace(np.log10(tmax * 1e-9), np.log10(tmax), NQ)
    a4 = 4 * alpha * tau
    s2 = a4 + 2 * sigma**2
    ex = -((w + v * tau)**2) / s2 - (z**2) / a4
    f = np.where(
        ex < -700, 0.0, np.exp(ex)
    ) / ((2 * alpha * tau + sigma**2) * np.sqrt(tau))
    val = trap(f, tau)
    return q / (np.pi * rho * cp * np.sqrt(4 * np.pi * alpha)) * val

def et_depth_fast(P, v, beamD, rho, cp, k, Tm,
                  T0=293.0, eta=0.35, sigma_factor=0.25):
    vals = np.array([P, v, beamD, rho, cp, k, Tm, eta], dtype=float)
    if not np.all(np.isfinite(vals)):
        return np.nan
    if min(P, v, beamD, rho, cp, k, eta) <= 0:
        return np.nan
    if Tm <= T0:
        return np.nan

    sigma = sigma_factor * beamD
    dTm = Tm - T0

    ws = np.linspace(-6 * sigma, 2 * sigma, 15)
    surf = [et_dT_grid(w, 1e-9, P, v, sigma, rho, cp, k, eta) for w in ws]
    if max(surf) < dTm:
        return 0.0

    wh = ws[int(np.argmax(surf))]
    lo, hi = 1e-9, sigma

    for _ in range(40):
        if et_dT_grid(wh, hi, P, v, sigma, rho, cp, k, eta) < dTm:
            break
        hi *= 1.6
    else:
        return np.nan

    for _ in range(60):
        mid = 0.5 * (lo + hi)
        if et_dT_grid(wh, mid, P, v, sigma, rho, cp, k, eta) >= dTm:
            lo = mid
        else:
            hi = mid

    return 0.5 * (lo + hi)

# ---------- nominal E-T predictions ----------
candidate = (~mt_bad) & (~beam_bad)

et = np.full(len(s), np.nan)
t0 = time.time()
idx = np.where(candidate.values)[0]

for j, i in enumerate(idx, 1):
    r = s.iloc[i]
    et[i] = et_depth_fast(
        P=float(r["Power"]) if pd.notna(r["Power"]) else np.nan,
        v=float(r["Velocity"]) / 1000.0 if pd.notna(r["Velocity"]) else np.nan, # mm/s -> m/s
        beamD=float(r["beam D"]) * 1e-6 if pd.notna(r["beam D"]) else np.nan,  # µm -> m
        rho=float(r["density"]) if pd.notna(r["density"]) else np.nan,
        cp=float(r["Cp"]) if pd.notna(r["Cp"]) else np.nan,
        k=float(r["k"]) if pd.notna(r["k"]) else np.nan,
        Tm=float(r["melting T"]) if pd.notna(r["melting T"]) else np.nan,
        T0=293.0,
        eta=float(eta_used[i]),
        sigma_factor=0.25,
    )
    if j % 200 == 0:
        print(f"  E-T: {j}/{len(idx)} rows")

print(f"E-T numerical evaluation: {(time.time()-t0):.1f} s")

compatible = candidate & np.isfinite(et) & (et > 0)
d = s.loc[compatible].reset_index(drop=True)
et_um = et[compatible.values] * 1e6  # et_depth_fast returns metres; convert to µm

print("\nAnalytical-comparison subset")
print("  rows   :", len(d))
print("  studies:", d["paper ID"].nunique())
print("  alloys :", d["Material"].nunique())

# Save compatible row audit.
compat_out = d[["Material","Process","Sub-process","paper ID","depth of meltpool"]].copy()
compat_out["ET_raw_depth_um"] = et_um
compat_out.to_csv(f"{RES}/eagar_tsai_compatible_rows.csv", index=False)

X = d[F3_REG].copy()
y = np.log10(d["depth of meltpool"].astype(float).values)
et_log = np.log10(et_um)
g = d["paper ID"].astype(str).values

# ---------- protocol definitions on the compatible subset ----------
protocols = {}

v0 = np.full(len(d), -1, dtype=int)
for f, (_, te) in enumerate(
    KFold(n_splits=10, shuffle=True, random_state=SEED).split(np.arange(len(d)))
):
    v0[te] = f
protocols["V0_random"] = (v0, False)

v1 = np.full(len(d), -1, dtype=int)
for f, (_, te) in enumerate(
    GroupKFold(n_splits=10).split(np.arange(len(d)), groups=g)
):
    v1[te] = f
protocols["V1_by_study"] = (v1, True)

# Keep the same LOMO material definition as the main frozen benchmark.
v2 = np.full(len(d), -1, dtype=int)
for f, mat in enumerate(META["regression"]["lomo_depth_materials"]):
    v2[d["Material"].values == mat] = f
protocols["V2_by_alloy"] = (v2, True)

# ---------- manual nested tuning (same search spaces as Notebook 03) ----------
RF_GRID = [
    {"max_depth": md, "min_samples_leaf": leaf}
    for md in [None, 10]
    for leaf in [1, 3]
]
XGB_GRID = [
    {"max_depth": md, "learning_rate": lr, "subsample": 0.8}
    for md in [3, 6]
    for lr in [0.05, 0.10]
]

def make_regressor(name, params):
    if name == "RF":
        return Pipeline([
            ("imp", SimpleImputer(strategy="median")),
            ("m", RandomForestRegressor(
                random_state=SEED,
                n_estimators=300,
                max_depth=params["max_depth"],
                min_samples_leaf=params["min_samples_leaf"],
                n_jobs=-1
            ))
        ])
    if name == "XGB":
        return XGBRegressor(
            random_state=SEED,
            n_estimators=400,
            max_depth=params["max_depth"],
            learning_rate=params["learning_rate"],
            subsample=params["subsample"],
            verbosity=0,
            n_jobs=-1
        )
    raise ValueError(name)

def tune_regressor(name, Xtr, ytr, gtr=None):
    grid = RF_GRID if name == "RF" else XGB_GRID

    if gtr is not None and len(np.unique(gtr)) >= 3:
        splitter = list(
            GroupKFold(n_splits=3).split(np.arange(len(ytr)), groups=gtr)
        )
    else:
        splitter = list(
            KFold(n_splits=3, shuffle=True, random_state=SEED)
            .split(np.arange(len(ytr)))
        )

    best_mae = np.inf
    best_params = None

    for params in grid:
        oof = np.full(len(ytr), np.nan)

        for itr, ite in splitter:
            mdl = make_regressor(name, params)
            mdl.fit(Xtr.iloc[itr], ytr[itr])
            oof[ite] = mdl.predict(Xtr.iloc[ite])

        mae = mean_absolute_error(ytr, oof)
        if mae < best_mae:
            best_mae = mae
            best_params = params.copy()

    return best_params, float(best_mae)

# ---------- outer OOF comparison ----------
prediction_frames = []
comparison_rows = []
tuning_rows = []

for pname, (fold_ids, grouped_inner) in protocols.items():
    pred_cal = np.full(len(d), np.nan)
    pred_rf = np.full(len(d), np.nan)
    pred_xgb = np.full(len(d), np.nan)

    folds = sorted(int(x) for x in np.unique(fold_ids) if x >= 0)
    print(f"\n=== {pname}: {len(folds)} outer folds ===")

    for f in folds:
        te = fold_ids == f
        tr = (fold_ids != f) & (fold_ids >= 0)

        if te.sum() == 0 or tr.sum() < 20:
            continue

        # Two-parameter log-space calibration of E-T on OUTER training data only.
        slope, intercept = np.polyfit(et_log[tr], y[tr], 1)
        pred_cal[te] = intercept + slope * et_log[te]

        gtr = g[tr] if grouped_inner else None

        for model_name, dest in [("RF", pred_rf), ("XGB", pred_xgb)]:
            best_params, inner_mae = tune_regressor(
                model_name,
                X.loc[tr].reset_index(drop=True),
                y[tr],
                gtr=gtr
            )

            mdl = make_regressor(model_name, best_params)
            mdl.fit(X.loc[tr], y[tr])
            dest[te] = mdl.predict(X.loc[te])

            tuning_rows.append({
                "protocol": pname,
                "outer_fold": f,
                "model": model_name,
                "n_train": int(tr.sum()),
                "n_test": int(te.sum()),
                "inner_MAE_log": inner_mae,
                "best_params": json.dumps(best_params),
            })

        print(f"  fold {f:02d}: train={tr.sum():4d}, test={te.sum():4d}")

    eval_mask = (
        (fold_ids >= 0) &
        np.isfinite(pred_cal) &
        np.isfinite(pred_rf) &
        np.isfinite(pred_xgb)
    )

    yy = y[eval_mask]
    raw = et_log[eval_mask]
    cal = pred_cal[eval_mask]
    rf = pred_rf[eval_mask]
    xg = pred_xgb[eval_mask]

    row = {
        "protocol": pname,
        "n": int(eval_mask.sum()),
        "n_studies": int(pd.Series(g[eval_mask]).nunique()),
        "R2_rawET": float(r2_score(yy, raw)),
        "R2_calET": float(r2_score(yy, cal)),
        "R2_RF": float(r2_score(yy, rf)),
        "R2_XGB": float(r2_score(yy, xg)),
        "RF_advantage_R2": float(r2_score(yy, rf) - r2_score(yy, cal)),
        "XGB_advantage_R2": float(r2_score(yy, xg) - r2_score(yy, cal)),
    }
    comparison_rows.append(row)

    pf = d.loc[eval_mask, ["paper ID","Material","depth of meltpool"]].copy()
    pf["protocol"] = pname
    pf["y_true_log"] = yy
    pf["ET_raw_log"] = raw
    pf["ET_cal_log"] = cal
    pf["RF_log"] = rf
    pf["XGB_log"] = xg
    prediction_frames.append(pf)

comparison = pd.DataFrame(comparison_rows)
pred_all = pd.concat(prediction_frames, ignore_index=True)

# ---------- source-level bootstrap of ML - calibrated E-T R² ----------
def source_bootstrap_advantage(frame, pred_col, B=2000, seed=SEED):
    rng = np.random.default_rng(seed)
    src = frame["paper ID"].astype(str).values
    unique_src = np.unique(src)
    src_to_idx = {s0: np.where(src == s0)[0] for s0 in unique_src}

    vals = []
    for _ in range(B):
        sampled = rng.choice(unique_src, size=len(unique_src), replace=True)
        idx = np.concatenate([src_to_idx[s0] for s0 in sampled])

        yy = frame["y_true_log"].values[idx]
        etp = frame["ET_cal_log"].values[idx]
        mlp = frame[pred_col].values[idx]

        if np.nanstd(yy) <= 0:
            continue
        vals.append(r2_score(yy, mlp) - r2_score(yy, etp))

    return float(np.percentile(vals, 2.5)), float(np.percentile(vals, 97.5)), len(vals)

boot_rows = []
for pname in comparison["protocol"]:
    frame = pred_all[pred_all["protocol"].eq(pname)].reset_index(drop=True)
    for model, col in [("RF","RF_log"), ("XGB","XGB_log")]:
        lo, hi, nb = source_bootstrap_advantage(frame, col, B=2000)
        est = float(
            comparison.loc[comparison["protocol"].eq(pname), f"{model}_advantage_R2"].iloc[0]
        )
        boot_rows.append({
            "protocol": pname,
            "model": model,
            "advantage_R2": est,
            "CI95_lo": lo,
            "CI95_hi": hi,
            "bootstrap_replicates_used": nb,
        })

boot = pd.DataFrame(boot_rows)

# Merge printable CIs into main result.
for model in ["RF","XGB"]:
    tmp = boot[boot["model"].eq(model)][
        ["protocol","CI95_lo","CI95_hi"]
    ].rename(columns={
        "CI95_lo": f"{model}_adv_CI95_lo",
        "CI95_hi": f"{model}_adv_CI95_hi",
    })
    comparison = comparison.merge(tmp, on="protocol", how="left")

comparison.to_csv(f"{RES}/eagar_tsai_comparison_corrected.csv", index=False)
boot.to_csv(f"{RES}/eagar_tsai_bootstrap_corrected.csv", index=False)
pred_all.to_csv(f"{RES}/eagar_tsai_oof_predictions_corrected.csv", index=False)
pd.DataFrame(tuning_rows).to_csv(
    f"{RES}/eagar_tsai_nested_tuning_corrected.csv", index=False
)

print("\nCORRECTED E-T / TREE-ENSEMBLE COMPARISON")
print(comparison.round(3).to_string(index=False))

print("\nSOURCE-LEVEL BOOTSTRAP")
print(boot.round(3).to_string(index=False))

# ---------- final figure ----------
order = ["V0_random","V1_by_study","V2_by_alloy"]
plot = comparison.set_index("protocol").loc[order].reset_index()
x = np.arange(len(order))
labels = ["V0\nrandom", "V1\nunseen study", "V2\nunseen alloy"]

fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))

ax[0].plot(x, plot["R2_calET"], marker="s", label="Eagar–Tsai (calibrated)")
ax[0].plot(x, plot["R2_RF"], marker="o", label="Random forest")
ax[0].plot(x, plot["R2_XGB"], marker="o", label="XGBoost")
ax[0].plot(x, plot["R2_rawET"], marker="^", linestyle="--", label="Eagar–Tsai (raw)")
ax[0].axhline(0, linewidth=0.8, linestyle=":")
ax[0].set_xticks(x)
ax[0].set_xticklabels(labels)
ax[0].set_ylabel("R² (log₁₀ depth)")
ax[0].set_title("(a) Tree ensembles versus analytical physics")
ax[0].grid(alpha=0.3)
ax[0].legend(fontsize=8)

width = 0.34
for j, model in enumerate(["RF","XGB"]):
    vals = plot[f"{model}_advantage_R2"].values
    lo = plot[f"{model}_adv_CI95_lo"].values
    hi = plot[f"{model}_adv_CI95_hi"].values
    ypos = x + (j - 0.5) * width
    yerr = np.vstack([vals - lo, hi - vals])

    ax[1].bar(ypos, vals, width=width, label=model)
    ax[1].errorbar(
        ypos, vals, yerr=yerr,
        fmt="none", ecolor="black", capsize=3, linewidth=1
    )

ax[1].axhline(0, linewidth=0.8, linestyle=":")
ax[1].set_xticks(x)
ax[1].set_xticklabels(labels)
ax[1].set_ylabel("ML advantage over calibrated E-T (ΔR²)")
ax[1].set_title("(b) Advantage depends on evaluation protocol")
ax[1].grid(axis="y", alpha=0.3)
ax[1].legend(fontsize=8)

plt.tight_layout()
plt.savefig(f"{RES}/fig_eagar_tsai_corrected.pdf", bbox_inches="tight")
plt.savefig(f"{RES}/fig_eagar_tsai_corrected.png", dpi=600, bbox_inches="tight")
plt.show()

print("\nSaved corrected E-T outputs to:", RES)


In [ ]:

# ============================================================
# CELL 5 — EAGAR–TSAI SENSITIVITY GRID (OPTIONAL BUT REQUIRED
#          IF THE MANUSCRIPT RETAINS THE SENSITIVITY CLAIM)
# ============================================================
# Tests sigma = D/2, D/4, D/8 and absorptivity defaults 0.25, 0.35, 0.45.
# Valid measured absorptivity is retained; the default is used only when the
# benchmark absorptivity is missing or outside (0,1].

SIGMA_FACTORS = [0.50, 0.25, 0.125]
ETA_DEFAULTS = [0.25, 0.35, 0.45]

# Use the same pre-audited source table from Cell 4.
s0 = reg[reg["has_depth"]].reset_index(drop=True).copy()
mt_bad0 = melting_temperature_bad_mask(s0)
beam0 = pd.to_numeric(s0["beam D"], errors="coerce")
beam_bad0 = beam0.notna() & (beam0 < 1)
base_candidate = (~mt_bad0) & (~beam_bad0)

a0 = pd.to_numeric(s0["absorption coefficient"], errors="coerce")
valid_a0 = (a0 > 0) & (a0 <= 1)

sens_rows = []
t0 = time.time()

for sf in SIGMA_FACTORS:
    for eta_default in ETA_DEFAULTS:
        eta0 = np.where(valid_a0, a0, eta_default)
        et0 = np.full(len(s0), np.nan)

        idx0 = np.where(base_candidate.values)[0]
        for i in idx0:
            r = s0.iloc[i]
            et0[i] = et_depth_fast(
                P=float(r["Power"]) if pd.notna(r["Power"]) else np.nan,
                v=float(r["Velocity"]) / 1000.0 if pd.notna(r["Velocity"]) else np.nan,
                beamD=float(r["beam D"]) * 1e-6 if pd.notna(r["beam D"]) else np.nan,
                rho=float(r["density"]) if pd.notna(r["density"]) else np.nan,
                cp=float(r["Cp"]) if pd.notna(r["Cp"]) else np.nan,
                k=float(r["k"]) if pd.notna(r["k"]) else np.nan,
                Tm=float(r["melting T"]) if pd.notna(r["melting T"]) else np.nan,
                T0=293.0,
                eta=float(eta0[i]),
                sigma_factor=sf,
            )

        ok = base_candidate & np.isfinite(et0) & (et0 > 0)
        ds = s0.loc[ok].reset_index(drop=True)
        et_s = et0[ok.values] * 1e6  # convert metres to µm before log-space calibration

        yy = np.log10(ds["depth of meltpool"].astype(float).values)
        eel = np.log10(et_s)
        gg = ds["paper ID"].astype(str).values

        # V1 only for sensitivity: source-held-out GroupKFold.
        ff = np.full(len(ds), -1, dtype=int)
        nsp = min(10, len(np.unique(gg)))
        for f, (_, te) in enumerate(
            GroupKFold(n_splits=nsp).split(np.arange(len(ds)), groups=gg)
        ):
            ff[te] = f

        cal = np.full(len(ds), np.nan)
        for f in sorted(np.unique(ff)):
            te = ff == f
            tr = ff != f
            if te.sum() == 0 or tr.sum() < 20:
                continue
            slope, intercept = np.polyfit(eel[tr], yy[tr], 1)
            cal[te] = intercept + slope * eel[te]

        m = np.isfinite(cal)
        sens_rows.append({
            "sigma_factor_D": sf,
            "eta_default": eta_default,
            "n": int(m.sum()),
            "n_studies": int(pd.Series(gg[m]).nunique()),
            "R2_calET_V1": float(r2_score(yy[m], cal[m])),
        })

        print(
            f"sigma=D*{sf:<5g} eta_default={eta_default:.2f} "
            f"n={m.sum():4d} R2_calET_V1={sens_rows[-1]['R2_calET_V1']:.3f}"
        )

sens = pd.DataFrame(sens_rows)
sens.to_csv(f"{RES}/eagar_tsai_sensitivity_corrected.csv", index=False)

print(f"\nSensitivity grid completed in {(time.time()-t0)/60:.1f} min")
print("\nRange of calibrated E-T V1 R²:")
print(
    f"  {sens['R2_calET_V1'].min():.3f} to "
    f"{sens['R2_calET_V1'].max():.3f}"
)
print("\nSaved:", f"{RES}/eagar_tsai_sensitivity_corrected.csv")


In [ ]:

# ============================================================
# CELL 6 — PACKAGE ALL FINAL RESULTS FOR DOWNLOAD
# ============================================================
import shutil

archive = "/content/Paper4_final_validation_results"
shutil.make_archive(archive, "zip", RES)

print("Created:")
print(archive + ".zip")
print("\nDownload this ZIP from the Colab Files panel and upload it to ChatGPT.")
print("\nFiles generated:")
for f in sorted(os.listdir(RES)):
    print(" ", f)
